In [2]:
import pandas as pd
import os

# --- CONFIGURATION ---
radiomics_stats_file = '../Results/table_statistical_analysis.csv' 
dosiomics_file = '../Results/final_local_dosiomics.csv'
output_table = '../Results/Table_3_1_Formatted.csv'

print("--- GENERATING TABLE 3.1 AUTOMATICALLY ---")

# 1. Load Radiomics Stats (Already calculated in LUNG-20)
if os.path.exists(radiomics_stats_file):
    df_rad_stats = pd.read_csv(radiomics_stats_file)
    print("Radiomics stats loaded.")
else:
    print("Error: Radiomics stats file missing. Run LUNG-20 first.")
    df_rad_stats = pd.DataFrame()

# 2. Calculate Dosiomics Stats (On the fly from LUNG-18 data)
# We need to calculate Mean +/- SD and P-value for Dosiomics since LUNG-20 was Radiomics only
dosiomics_rows = []
if os.path.exists(dosiomics_file):
    df_dos = pd.read_csv(dosiomics_file)
    print("Dosiomics data loaded.")
    
    from scipy.stats import mannwhitneyu
    
    # Define metrics to check
    metrics = ['D95_Gy', 'Mean_Dose_Gy', 'V20Gy_%']
    
    group1 = df_dos[df_dos['Source'] == 'Square']
    group2 = df_dos[df_dos['Source'] == 'Ahsania']
    
    for m in metrics:
        v1 = group1[m].dropna()
        v2 = group2[m].dropna()
        
        mean1, sd1 = v1.mean(), v1.std()
        mean2, sd2 = v2.mean(), v2.std()
        
        # Mann-Whitney Test
        stat, p = mannwhitneyu(v1, v2)
        
        # Determine "Drift Detected?"
        drift = "Yes (epsilon)" if p < 0.05 else "No"
        if p < 0.001: drift = "Yes (Delta)"
        
        dosiomics_rows.append({
            'Feature Category': 'Dosiomic',
            'Parameter': m,
            'Source Domain (Mean ± SD)': f"{mean1:.1f} ± {sd1:.1f}", # Square as Source/Reference here
            'Local Cohort (Mean ± SD)': f"{mean2:.1f} ± {sd2:.1f}",  # Ahsania as Local/Test
            'p-value': f"{p:.3f}",
            'Drift Detected?': drift
        })

# 3. Format Radiomics Rows (Selecting key features)
radiomics_rows = []
if not df_rad_stats.empty:
    # Pick top 2 most significant features
    top_feats = df_rad_stats.sort_values(by='P_Value').head(2)
    
    for _, row in top_feats.iterrows():
        # Clean name
        clean_name = row['Feature'].replace('original_', '').replace('glcm_', '').replace('firstorder_', '')[:20]
        
        # Determine drift type
        p = float(row['P_Value'])
        drift = "Yes (epsilon)" if p < 0.05 else "No"
        if p < 0.001: drift = "Yes (Delta)"

        radiomics_rows.append({
            'Feature Category': 'Radiomic',
            'Parameter': clean_name,
            'Source Domain (Mean ± SD)': row['Mean (Public (Western))'],
            'Local Cohort (Mean ± SD)': row['Mean (Local (Square Hospital))'],
            'p-value': f"{p:.3f}" if p >= 0.001 else "< 0.001",
            'Drift Detected?': drift
        })

# 4. Combine and Save
all_rows = dosiomics_rows + radiomics_rows
df_table = pd.DataFrame(all_rows)

print("\n--- FINAL TABLE 3.1 PREVIEW ---")
display(df_table)

df_table.to_csv(output_table, index=False)
print(f"\nSaved to: {os.path.abspath(output_table)}")

--- GENERATING TABLE 3.1 AUTOMATICALLY ---
Radiomics stats loaded.
Dosiomics data loaded.


KeyError: 'V20Gy_%'

In [3]:
import pandas as pd
import os
from scipy.stats import mannwhitneyu

# --- CONFIGURATION ---
radiomics_stats_file = '../Results/table_statistical_analysis.csv' 
dosiomics_file = '../Results/final_local_dosiomics.csv'
output_table = '../Results/Table_4_1_Formatted.csv' # Changed to 4.1 to match your request

print("--- GENERATING TABLE 4.1 AUTOMATICALLY ---")

# 1. Load Radiomics Stats (Already calculated in LUNG-20)
if os.path.exists(radiomics_stats_file):
    df_rad_stats = pd.read_csv(radiomics_stats_file)
    print("Radiomics stats loaded.")
else:
    print("Error: Radiomics stats file missing. Run LUNG-20 first.")
    df_rad_stats = pd.DataFrame()

# 2. Calculate Dosiomics Stats (On the fly from LUNG-18 data)
dosiomics_rows = []
if os.path.exists(dosiomics_file):
    df_dos = pd.read_csv(dosiomics_file)
    print("Dosiomics data loaded.")
    
    # Define metrics to check
    metrics = ['D95_Gy', 'Mean_Dose_Gy', 'V20Gy_%']
    
    # Filter groups based on Source column
    group1 = df_dos[df_dos['Source'] == 'Square']
    group2 = df_dos[df_dos['Source'] == 'Ahsania']
    
    for m in metrics:
        # Get values, dropping NaNs to avoid errors
        # NOTE: Using .get() with default to handle missing columns gracefully if needed, 
        # but direct access is better if we expect columns to exist.
        # The KeyError 'V20Gy_%' suggests the column name in df_dos might be slightly different.
        # Let's check columns first or strip whitespace.
        
        # Robust column finding
        col_name = None
        for c in df_dos.columns:
            if m.lower() in c.lower():
                col_name = c
                break
        
        if col_name is None:
            print(f"Warning: Metric {m} not found in columns: {df_dos.columns}")
            continue

        v1 = group1[col_name].dropna()
        v2 = group2[col_name].dropna()
        
        if len(v1) == 0 or len(v2) == 0:
            print(f"Warning: No data for {m}")
            continue

        mean1, sd1 = v1.mean(), v1.std()
        mean2, sd2 = v2.mean(), v2.std()
        
        # Mann-Whitney Test
        try:
            stat, p = mannwhitneyu(v1, v2)
            
            # Determine "Drift Detected?"
            drift = "Yes (epsilon)" if p < 0.05 else "No"
            if p < 0.001: drift = "Yes (Delta)"
            
            dosiomics_rows.append({
                'Feature Category': 'Dosiomic',
                'Parameter': m,
                'Source Domain (Mean ± SD)': f"{mean1:.1f} ± {sd1:.1f}", # Square
                'Local Cohort (Mean ± SD)': f"{mean2:.1f} ± {sd2:.1f}",  # Ahsania
                'p-value': f"{p:.3f}",
                'Drift Detected?': drift
            })
        except Exception as e:
             print(f"Stats error for {m}: {e}")

# 3. Format Radiomics Rows (Selecting key features)
radiomics_rows = []
if not df_rad_stats.empty:
    # Pick top 2 most significant features
    # Check if 'P_Value' column exists
    if 'P_Value' in df_rad_stats.columns:
        top_feats = df_rad_stats.sort_values(by='P_Value').head(2)
        
        for _, row in top_feats.iterrows():
            # Clean name
            clean_name = row['Feature'].replace('original_', '').replace('glcm_', '').replace('firstorder_', '')[:20]
            
            # Determine drift type
            p = float(row['P_Value'])
            drift = "Yes (epsilon)" if p < 0.05 else "No"
            if p < 0.001: drift = "Yes (Delta)"

            # Handle potentially missing columns in stats file depending on how LUNG-20 saved them
            # Assuming standard naming from previous steps
            mean_public = row.get('Mean (Public (Western))', 'N/A')
            mean_local = row.get('Mean (Local (Square Hospital))', 'N/A')

            radiomics_rows.append({
                'Feature Category': 'Radiomic',
                'Parameter': clean_name,
                'Source Domain (Mean ± SD)': mean_public,
                'Local Cohort (Mean ± SD)': mean_local,
                'p-value': f"{p:.3f}" if p >= 0.001 else "< 0.001",
                'Drift Detected?': drift
            })

# 4. Combine and Save
all_rows = dosiomics_rows + radiomics_rows
if all_rows:
    df_table = pd.DataFrame(all_rows)
    print("\n--- FINAL TABLE PREVIEW ---")
    print(df_table)
    df_table.to_csv(output_table, index=False)
    print(f"\nSaved to: {os.path.abspath(output_table)}")
else:
    print("No rows generated.")

--- GENERATING TABLE 4.1 AUTOMATICALLY ---
Radiomics stats loaded.
Dosiomics data loaded.
       'Max_Dose_Gy', 'Min_Dose_Gy', 'D95_Gy'],
      dtype='object')

--- FINAL TABLE PREVIEW ---
  Feature Category             Parameter Source Domain (Mean ± SD)  \
0         Dosiomic                D95_Gy               48.4 ± 14.1   
1         Dosiomic          Mean_Dose_Gy               52.2 ± 11.8   
2         Radiomic      shape_Sphericity               0.40 ± 0.05   
3         Radiomic  shape_Maximum3DDiame            312.71 ± 52.93   

  Local Cohort (Mean ± SD)  p-value Drift Detected?  
0              54.6 ± 11.8    0.000     Yes (Delta)  
1               57.1 ± 9.8    0.000     Yes (Delta)  
2                      N/A  < 0.001     Yes (Delta)  
3                      N/A  < 0.001     Yes (Delta)  

Saved to: d:\Thesis_Project\Results\Table_4_1_Formatted.csv


In [4]:
import pandas as pd
import os

# --- CONFIGURATION ---
output_table = '../Results/Table_4_1_Baseline_Characteristics.csv'
os.makedirs('../Results', exist_ok=True)

print("--- GENERATING TABLE 4.1 (TEMPLATE) ---")

# Define the data structure
# NOTE: YOU MUST REPLACE THESE VALUES WITH YOUR ACTUAL CALCULATED STATS
# I have put realistic placeholders based on your project description.

data = [
    {
        'Feature': 'Sample Size (n)',
        'Public (n=422)': '422',
        'Square (n=121)': '121',
        'Ahsania (n=54)': '54'
    },
    {
        'Feature': 'Age (Years)',
        'Public (n=422)': '68.2 ± 9.4',  # Typical Western NSCLC age
        'Square (n=121)': '58.5 ± 11.2', # Typically younger in South Asia
        'Ahsania (n=54)': '56.1 ± 10.8'
    },
    {
        'Feature': 'Gender (Male %)',
        'Public (n=422)': '289 (68.5%)',
        'Square (n=121)': '98 (81.0%)', # Higher male prev in Bangladesh due to smoking
        'Ahsania (n=54)': '45 (83.3%)'
    },
    {
        'Feature': 'Gender (Female %)',
        'Public (n=422)': '133 (31.5%)',
        'Square (n=121)': '23 (19.0%)',
        'Ahsania (n=54)': '9 (16.7%)'
    },
    {
        'Feature': 'Clinical Stage',
        'Public (n=422)': '',
        'Square (n=121)': '',
        'Ahsania (n=54)': ''
    },
    {
        'Feature': '  - Stage I/II',
        'Public (n=422)': '185 (43.8%)',
        'Square (n=121)': '15 (12.4%)', # Typically lower early detection
        'Ahsania (n=54)': '5 (9.3%)'
    },
    {
        'Feature': '  - Stage III/IV',
        'Public (n=422)': '237 (56.2%)',
        'Square (n=121)': '106 (87.6%)', # Most patients present late
        'Ahsania (n=54)': '49 (90.7%)'
    },
    {
        'Feature': 'CT Scanner Model',
        'Public (n=422)': 'Siemens Somatom',
        'Square (n=121)': 'GE LightSpeed RT16',
        'Ahsania (n=54)': 'Mixed (Siemens/Unknown)'
    },
    {
        'Feature': 'Slice Thickness (mm)',
        'Public (n=422)': '3.0 mm (Fixed)',
        'Square (n=121)': '2.5 - 5.0 mm',
        'Ahsania (n=54)': '3.0 - 5.0 mm'
    },
    {
        'Feature': 'TPS Software',
        'Public (n=422)': 'Unknown / Mixed',
        'Square (n=121)': 'Varian Eclipse',
        'Ahsania (n=54)': 'Elekta Monaco 5.10'
    }
]

# Create DataFrame
df_table = pd.DataFrame(data)

# Display
print("\n--- TABLE 4.1 PREVIEW ---")
display(df_table)

# Save
df_table.to_csv(output_table, index=False)
print(f"\nSaved to: {os.path.abspath(output_table)}")
print("INSTRUCTIONS: Open this CSV in Excel. Update the numbers with your exact calculations.")

--- GENERATING TABLE 4.1 (TEMPLATE) ---

--- TABLE 4.1 PREVIEW ---


,Feature,Public (n=422),Square (n=121),Ahsania (n=54)
0,Sample Size (n),422,121,54
1,Age (Years),68.2 ± 9.4,58.5 ± 11.2,56.1 ± 10.8
2,Gender (Male %),289 (68.5%),98 (81.0%),45 (83.3%)
3,Gender (Female %),133 (31.5%),23 (19.0%),9 (16.7%)
4,Clinical Stage,,,
5,- Stage I/II,185 (43.8%),15 (12.4%),5 (9.3%)
6,- Stage III/IV,237 (56.2%),106 (87.6%),49 (90.7%)
7,CT Scanner Model,Siemens Somatom,GE LightSpeed RT16,Mixed (Siemens/Unknown)
8,Slice Thickness (mm),3.0 mm (Fixed),2.5 - 5.0 mm,3.0 - 5.0 mm
9,TPS Software,Unknown / Mixed,Varian Eclipse,Elekta Monaco 5.10



Saved to: d:\Thesis_Project\Results\Table_4_1_Baseline_Characteristics.csv
INSTRUCTIONS: Open this CSV in Excel. Update the numbers with your exact calculations.


In [5]:
import pandas as pd
import os

# --- CONFIGURATION ---
output_table = '../Results/Table_4_1_Baseline_Characteristics.csv'
os.makedirs('../Results', exist_ok=True)

print("--- GENERATING TABLE 4.1 (DEMOGRAPHICS) ---")

# ---------------------------------------------------------
# INSTRUCTIONS:
# Replace the text inside the quotes '' with your ACTUAL data 
# from your hospital Excel sheets.
# ---------------------------------------------------------

data = [
    {
        'Feature': 'Sample Size (n)',
        'Public (n=422)': '422',
        'Square (n=121)': '121',
        'Ahsania (n=54)': '54'
    },
    {
        'Feature': 'Age (Years)',
        # Format: Mean ± SD
        'Public (n=422)': '68.2 ± 9.4',  
        'Square (n=121)': '58.5 ± 11.2', 
        'Ahsania (n=54)': '56.1 ± 10.8'
    },
    {
        'Feature': 'Gender (Male %)',
        # Format: Count (Percentage%)
        'Public (n=422)': '289 (68.5%)',
        'Square (n=121)': '98 (81.0%)',
        'Ahsania (n=54)': '45 (83.3%)'
    },
    {
        'Feature': 'Gender (Female %)',
        'Public (n=422)': '133 (31.5%)',
        'Square (n=121)': '23 (19.0%)',
        'Ahsania (n=54)': '9 (16.7%)'
    },
    {
        # Section Header Row
        'Feature': 'Clinical Stage',
        'Public (n=422)': '',
        'Square (n=121)': '',
        'Ahsania (n=54)': ''
    },
    {
        'Feature': '  - Stage I/II',
        'Public (n=422)': '185 (43.8%)',
        'Square (n=121)': '15 (12.4%)',
        'Ahsania (n=54)': '5 (9.3%)'
    },
    {
        'Feature': '  - Stage III/IV',
        'Public (n=422)': '237 (56.2%)',
        'Square (n=121)': '106 (87.6%)',
        'Ahsania (n=54)': '49 (90.7%)'
    },
    {
        'Feature': 'CT Scanner Model',
        'Public (n=422)': 'Siemens Somatom',
        'Square (n=121)': 'GE LightSpeed RT16',
        'Ahsania (n=54)': 'Mixed / Unknown'
    },
    {
        'Feature': 'Slice Thickness',
        'Public (n=422)': '3.0 mm',
        'Square (n=121)': '2.5 - 5.0 mm',
        'Ahsania (n=54)': '3.0 - 5.0 mm'
    }
]

# Create DataFrame
df_table = pd.DataFrame(data)

# Display in Notebook
print("\n--- TABLE PREVIEW ---")
display(df_table)

# Save to CSV
df_table.to_csv(output_table, index=False)
print(f"\nSUCCESS! Table saved to: {os.path.abspath(output_table)}")
print("Copy the contents of this CSV into your Thesis Word Document.")

--- GENERATING TABLE 4.1 (DEMOGRAPHICS) ---

--- TABLE PREVIEW ---


,Feature,Public (n=422),Square (n=121),Ahsania (n=54)
0,Sample Size (n),422,121,54
1,Age (Years),68.2 ± 9.4,58.5 ± 11.2,56.1 ± 10.8
2,Gender (Male %),289 (68.5%),98 (81.0%),45 (83.3%)
3,Gender (Female %),133 (31.5%),23 (19.0%),9 (16.7%)
4,Clinical Stage,,,
5,- Stage I/II,185 (43.8%),15 (12.4%),5 (9.3%)
6,- Stage III/IV,237 (56.2%),106 (87.6%),49 (90.7%)
7,CT Scanner Model,Siemens Somatom,GE LightSpeed RT16,Mixed / Unknown
8,Slice Thickness,3.0 mm,2.5 - 5.0 mm,3.0 - 5.0 mm



SUCCESS! Table saved to: d:\Thesis_Project\Results\Table_4_1_Baseline_Characteristics.csv
Copy the contents of this CSV into your Thesis Word Document.


In [6]:
import pandas as pd
import os

# --- CONFIGURATION ---
output_table = '../Results/Table_4_1_Baseline_Characteristics.csv'
os.makedirs('../Results', exist_ok=True)

print("--- GENERATING TABLE 4.1 (TEMPLATE) ---")

# Define the data structure
# NOTE: YOU MUST REPLACE THESE VALUES WITH YOUR ACTUAL CALCULATED STATS
# I have put realistic placeholders based on your project description.

data = [
    {
        'Feature': 'Sample Size (n)',
        'Public (n=422)': '422',
        'Square (n=121)': '121',
        'Ahsania (n=54)': '54'
    },
    {
        'Feature': 'Age (Years)',
        'Public (n=422)': '68.2 ± 9.4',  # Typical Western NSCLC age
        'Square (n=121)': '58.5 ± 11.2', # Typically younger in South Asia
        'Ahsania (n=54)': '56.1 ± 10.8'
    },
    {
        'Feature': 'Gender (Male %)',
        'Public (n=422)': '289 (68.5%)',
        'Square (n=121)': '98 (81.0%)', # Higher male prev in Bangladesh due to smoking
        'Ahsania (n=54)': '45 (83.3%)'
    },
    {
        'Feature': 'Gender (Female %)',
        'Public (n=422)': '133 (31.5%)',
        'Square (n=121)': '23 (19.0%)',
        'Ahsania (n=54)': '9 (16.7%)'
    },
    {
        'Feature': 'Clinical Stage',
        'Public (n=422)': '',
        'Square (n=121)': '',
        'Ahsania (n=54)': ''
    },
    {
        'Feature': '  - Stage I/II',
        'Public (n=422)': '185 (43.8%)',
        'Square (n=121)': '15 (12.4%)', # Typically lower early detection
        'Ahsania (n=54)': '5 (9.3%)'
    },
    {
        'Feature': '  - Stage III/IV',
        'Public (n=422)': '237 (56.2%)',
        'Square (n=121)': '106 (87.6%)', # Most patients present late
        'Ahsania (n=54)': '49 (90.7%)'
    },
    {
        'Feature': 'CT Scanner Model',
        'Public (n=422)': 'Siemens Somatom',
        'Square (n=121)': 'GE LightSpeed RT16',
        'Ahsania (n=54)': 'Mixed (Siemens/Unknown)'
    },
    {
        'Feature': 'Slice Thickness (mm)',
        'Public (n=422)': '3.0 mm (Fixed)',
        'Square (n=121)': '2.5 - 5.0 mm',
        'Ahsania (n=54)': '3.0 - 5.0 mm'
    },
    {
        'Feature': 'TPS Software',
        'Public (n=422)': 'Unknown / Mixed',
        'Square (n=121)': 'Varian Eclipse',
        'Ahsania (n=54)': 'Elekta Monaco 5.10'
    }
]

# Create DataFrame
df_table = pd.DataFrame(data)

# Display
print("\n--- TABLE 4.1 PREVIEW ---")
display(df_table)

# Save
df_table.to_csv(output_table, index=False)
print(f"\nSaved to: {os.path.abspath(output_table)}")
print("INSTRUCTIONS: Open this CSV in Excel. Update the numbers with your exact calculations.")

--- GENERATING TABLE 4.1 (TEMPLATE) ---

--- TABLE 4.1 PREVIEW ---


,Feature,Public (n=422),Square (n=121),Ahsania (n=54)
0,Sample Size (n),422,121,54
1,Age (Years),68.2 ± 9.4,58.5 ± 11.2,56.1 ± 10.8
2,Gender (Male %),289 (68.5%),98 (81.0%),45 (83.3%)
3,Gender (Female %),133 (31.5%),23 (19.0%),9 (16.7%)
4,Clinical Stage,,,
5,- Stage I/II,185 (43.8%),15 (12.4%),5 (9.3%)
6,- Stage III/IV,237 (56.2%),106 (87.6%),49 (90.7%)
7,CT Scanner Model,Siemens Somatom,GE LightSpeed RT16,Mixed (Siemens/Unknown)
8,Slice Thickness (mm),3.0 mm (Fixed),2.5 - 5.0 mm,3.0 - 5.0 mm
9,TPS Software,Unknown / Mixed,Varian Eclipse,Elekta Monaco 5.10



Saved to: d:\Thesis_Project\Results\Table_4_1_Baseline_Characteristics.csv
INSTRUCTIONS: Open this CSV in Excel. Update the numbers with your exact calculations.
